# Surface Code: Threshold Analysis and Syndrome Heatmaps

This notebook walks through the core capabilities of `surfacecode`:

1. Run a single surface-code memory experiment and read off its logical error rate.
2. Visualize the **syndrome heatmap** (where/when detectors fire).
3. Sweep code distance against physical error rate to estimate the **threshold**.
4. Show **logical error rate vs distance** below threshold.

The whole stack is built on [Stim](https://github.com/quantumlib/Stim) (circuit simulation) and
[PyMatching](https://pymatching.readthedocs.io/) (MWPM decoding).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from surfacecode import (
    ExperimentConfig,
    build_surface_code_circuit,
    run_memory_experiment,
    run_threshold_sweep,
    sample_syndromes,
)
from surfacecode.viz import (
    plot_logical_vs_distance,
    plot_syndrome_heatmap,
    plot_threshold_sweep,
)

## 1. A single memory experiment

We run a distance-5 rotated surface code for 5 rounds at a 0.1% physical error rate and decode
with MWPM. The result reports the per-shot logical error rate, a Wilson 95% interval, and the
per-round rate.

In [ ]:
config = ExperimentConfig(distance=5, rounds=5, p=0.001, shots=50_000, seed=2026)
result = run_memory_experiment(config)

print(f"logical error rate : {result.logical_error_rate:.4e}")
print(f"95% CI             : [{result.ci_low:.4e}, {result.ci_high:.4e}]")
print(f"per-round rate     : {result.per_round_error_rate:.4e}")
print(f"detectors / obs    : {result.num_detectors} / {result.num_observables}")

## 2. Syndrome heatmap

Each measurement round produces a layer of detectors (stabilizer parity checks). Plotting the
firing frequency of every detector reveals the spatial-temporal structure of detected errors.
We raise the error rate to 1% here so the pattern is clearly visible.

In [ ]:
heat_config = ExperimentConfig(distance=5, rounds=5, p=0.01, shots=20_000, seed=7)
circuit = build_surface_code_circuit(heat_config)
sample = sample_syndromes(circuit, shots=heat_config.shots, seed=heat_config.seed)

plot_syndrome_heatmap(sample, rounds=heat_config.rounds)
plt.tight_layout()
plt.show()

## 3. Threshold sweep

We sweep distances 3, 5, 7 across a range of physical error rates. Below threshold, larger codes
have *lower* logical error rates; above threshold the ordering flips. The crossing point is the
threshold estimate.

In [ ]:
sweep = run_threshold_sweep(
    distances=[3, 5, 7],
    error_rates=[0.005, 0.008, 0.01, 0.012, 0.015],
    shots=20_000,
    seed=2026,
)
print(f"threshold estimate p_th ~ {sweep.threshold_estimate:.4f}")

plot_threshold_sweep(sweep)
plt.tight_layout()
plt.show()

## 4. Logical error rate vs distance (below threshold)

Fixing the physical error rate well below threshold, the logical error rate should fall roughly
exponentially with code distance, the hallmark of a working error-correcting code.

In [ ]:
plot_logical_vs_distance(sweep, p=0.005)
plt.tight_layout()
plt.show()